# Training on Kaggle or Colab

A template for training one fold of MammoSeg on a free cloud GPU. It installs the
package, rebuilds the derived data from the INbreast dataset, and runs
`mammo-seg train`. Outputs are stripped before commit (`nbstripout`). Research use
only: this is not a medical device.

**Kaggle setup (once):**
- Add the INbreast dataset to the notebook as an input (*Add input*).
- Settings: accelerator *GPU* and *Internet on* (the repository and the pretrained
  encoder weights are downloaded).

**Sessions end** after at most about 12 hours, or earlier if idle. The training
stops cleanly before `MAX_MINUTES` and saves `last.pt` after every epoch. To
continue, save a version of the notebook, add its output as an input of the next
session, set `PREVIOUS_RUNS` below to its `runs` folder and run again: the notebook
resumes by itself when it finds a `last.pt`.

On Colab, put the dataset in a folder (for example from Google Drive), point
`DATASET_DIR` to it and set `WORK` to a folder that persists, such as one on Drive.
Never put `kaggle.json` or any other credential in a notebook.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/GiannisPapa97/mammo_seg.git"
WORK = Path("/kaggle/working")  # Colab: a folder on Google Drive
DATASET_DIR = None  # None: look for the INbreast folder in /kaggle/input
PREVIOUS_RUNS = None  # for example Path("/kaggle/input/my-notebook/runs")
FOLD = 0
MAX_MINUTES = 10 * 60  # stop cleanly before the session limit

## 1. Install

`.[cloud]` adds the model library and leaves the PyTorch of the notebook (a GPU
build) untouched.

In [ ]:
!test -d {WORK}/mammo_seg || git clone -q {REPO_URL} {WORK}/mammo_seg
!pip install -q "{WORK}/mammo_seg[cloud]"
!mammo-seg --help

In [ ]:
import torch

print(
    torch.__version__,
    "- GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
)

## 2. Derived data

Manifest, patient-level splits, masks and preprocessed images are rebuilt from the
raw dataset (a few minutes), so no derived data has to be uploaded. A step that is
already done in this folder is skipped.

In [ ]:
if DATASET_DIR is None:
    found = sorted(Path("/kaggle/input").rglob("INbreast.csv"))
    assert found, "attach the INbreast dataset as an input of the notebook"
    DATASET_DIR = found[0].parent
print("dataset:", DATASET_DIR)

PROCESSED = WORK / "data" / "processed"
RUNS = WORK / "runs"
MANIFEST = PROCESSED / "manifest.json"
SPLITS = PROCESSED / "splits.json"
MASKS = PROCESSED / "masks"
PREPROCESSED = PROCESSED / "preprocessed"
PROCESSED.mkdir(parents=True, exist_ok=True)

In [ ]:
if not MANIFEST.exists():
    !mammo-seg manifest --dataset-dir "{DATASET_DIR}" --output {MANIFEST}
if not SPLITS.exists():
    !mammo-seg splits --folds 5 --test-fraction 0.2 \
        --manifest {MANIFEST} --output {SPLITS}
if not MASKS.exists():
    !mammo-seg masks --manifest {MANIFEST} --output-dir {MASKS}
if not (PREPROCESSED / "preprocessing.json").exists():
    !mammo-seg preprocess --manifest {MANIFEST} --dataset-dir "{DATASET_DIR}" \
        --masks-dir {MASKS} --output-dir {PREPROCESSED}

## 3. Train

The checkpoints and `history.jsonl` go to `runs/fold_<FOLD>`. The best model by the
Dice over all validation images is `best.pt`; `--selection-metric` can choose
another metric (for example `with_mass.dice`).

In [ ]:
import shutil

fold_dir = RUNS / f"fold_{FOLD}"
if PREVIOUS_RUNS is not None and not fold_dir.exists():
    shutil.copytree(Path(PREVIOUS_RUNS) / f"fold_{FOLD}", fold_dir)
resume = "--resume" if (fold_dir / "last.pt").exists() else ""
print("resuming" if resume else "starting a new run")

In [ ]:
!mammo-seg train --manifest {MANIFEST} --splits {SPLITS} \
    --preprocessed-dir {PREPROCESSED} --output-dir {RUNS} --fold {FOLD} \
    --num-workers 2 --max-minutes {MAX_MINUTES} {resume}

## 4. Results

One line per epoch is in `history.jsonl`. Undefined metrics are `null`. The weights
of the best epoch are in `best.pt`; download it from the notebook output.

In [ ]:
import json

rows = [
    json.loads(line) for line in (fold_dir / "history.jsonl").read_text().splitlines()
]
for row in rows[-5:]:
    everything, positives = (
        row["validation"]["all_images"],
        row["validation"]["with_mass"],
    )
    print(
        row["epoch"],
        f"loss {row['train_loss']:.3f}",
        f"Dice {everything['dice']:.3f}",
        f"Dice on masses {positives['dice']}",
    )